## Load libraries

In [11]:
# Cell 1: Enhanced Imports with LangChain & Multi-LLM Framework
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import hashlib
import json
import pickle
import joblib
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# LangChain & LLM Libraries
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEndpoint, ChatHuggingFace
from langchain_classic.chains import LLMChain
from langchain_core.prompts import PromptTemplate

# HuggingFace & Groq
from huggingface_hub import InferenceClient
import groq
from groq import Groq

# Security
from cryptography.fernet import Fernet
import ast
import base64
import os
from dotenv import load_dotenv
from pathlib import Path

# Load environment variables
# Load env path
ENV_PATH = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=ENV_PATH)

print(f"✅ All libraries loaded at {datetime.now()}")

# Initialize security
SECURITY_KEY = Fernet.generate_key()
cipher = Fernet(SECURITY_KEY)
print(f"🔐 Security key generated: {SECURITY_KEY[:16]}...")

# Configure API Keys from environment or input
HUGGINGFACE_API_KEY = os.getenv('HUGGINGFACE_API_KEY')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

print(f"🔑 API Keys configured:")
print(f"  - HuggingFace: {'✅' if HUGGINGFACE_API_KEY else '❌'}")
print(f"  - Groq: {'✅' if GROQ_API_KEY else '❌'}")

✅ All libraries loaded at 2026-07-03 21:38:52.872175
🔐 Security key generated: b'NLRSc6kNJXV8NUeI'...
🔑 API Keys configured:
  - HuggingFace: ✅
  - Groq: ✅


## Load database

In [12]:
DATABASE = Path.cwd().parent / "app" / "data" / "cleaned" / "sql_supermarket.parquet"
pd.set_option('display.max_columns', None)
df = pd.read_parquet(DATABASE)
df

,id,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year,unit_price,profit_margin
0,1,AG-2011-2040,2011-01-01,2011-06-01,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,Africa,OFF-TEN-10000025,Office Supplies,Storage,"Tenex Lockers, Blue",4080000.0,2,0.0,1061400.0,354600.0,Medium,2011,2040000.0,0.26
1,2,IN-2011-47883,2011-01-01,2011-08-01,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,OFF-SU-10000618,Office Supplies,Supplies,"Acme Trimmer, High Speed",1200000.0,3,0.1,360360.0,97200.0,Medium,2011,400000.0,0.30
2,3,HU-2011-1220,2011-01-01,2011-05-01,Second Class,Annie Thurman,Consumer,Budapest,Hungary,EMEA,EMEA,OFF-TEN-10001585,Office Supplies,Storage,"Tenex Box, Single Width",660000.0,4,0.0,296400.0,81700.0,High,2011,165000.0,0.45
3,4,IT-2011-3647632,2011-01-01,2011-05-01,Second Class,Eugene Moren,Home Office,Stockholm,Sweden,EU,North,OFF-PA-10001492,Office Supplies,Paper,"Enermax Note Cards, Premium",450000.0,3,0.5,-260550.0,48200.0,High,2011,150000.0,-0.58
4,5,CA-2011-1510,2011-02-01,2011-06-01,Standard Class,Magdelene Morse,Consumer,Ontario,Canada,Canada,Canada,TEC-OKI-10002750,Technology,Machines,"Okidata Inkjet, Wireless",3140000.0,1,0.0,31200.0,241000.0,Medium,2011,3140000.0,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25030,25031,CA-2014-115427,2014-12-12,2015-04-01,Standard Class,Erica Bern,Corporate,California,United States,US,West,OFF-BI-10004632,Office Supplies,Binders,GBC Binding covers,210000.0,2,0.2,64750.0,20600.0,Medium,2014,105000.0,0.31
25031,25032,UP-2014-4410,2014-12-12,2015-04-01,Standard Class,Guy Thornton,Consumer,Zaporizhzhya,Ukraine,EMEA,EMEA,OFF-AVE-10003558,Office Supplies,Labels,"Avery Round Labels, Alphabetical",280000.0,4,0.0,61200.0,17000.0,Medium,2014,70000.0,0.22
25032,25033,MX-2014-108574,2014-12-12,2015-04-01,Standard Class,Julia Barnett,Home Office,Tamaulipas,Mexico,LATAM,North,OFF-LA-10004969,Office Supplies,Labels,"Novimex Legal Exhibit Labels, Adjustable",170000.0,3,0.0,6600.0,13200.0,Medium,2014,56700.0,0.04
25033,25034,MO-2014-2560,2014-12-12,2015-05-01,Standard Class,Liz Preis,Consumer,Souss-Massa-Draâ,Morocco,Africa,Africa,OFF-WIL-10001069,Office Supplies,Binders,"Wilson Jones Hole Reinforcements, Clear",40000.0,1,0.0,4200.0,4900.0,Medium,2014,40000.0,0.10


## Multi LLM Manager with Failover Support

In [ ]:
from abc import ABC, abstractmethod
from typing import Optional, Dict, Any, List
import time
import requests

class LLMProvider(ABC):
    """Abstract base class for LLM providers."""

    @abstractmethod
    def generate_response(self, prompt: str, **kwargs) -> str:
        pass

    @abstractmethod
    def is_available(self) -> bool:
        pass

class HuggingFaceProvider(LLMProvider):
    """HuggingFace Inference API provider."""

    def __init__(self, api_key: str, model: str = "mistralai/Mistral-7B-Instruct-v0.1"):
        self.api_key = api_key
        self.model = model
        self.client = InferenceClient(token=self.api_key)
        self._available = bool(api_key)

    def generate_response(self, prompt: str, **kwargs) -> str:
        try:
            response = self.client.text_generation(
                prompt=prompt,
                model=self.model,
                max_new_tokens=kwargs.get("max_new_tokens", 256),
                temperature=kwargs.get("temperature", 0.3),
                top_p=kwargs.get("top_p", 0.9)
            )
            return response
        
        except Exception as e:
            print(f"Error generating response from HuggingFace: {e}")
            raise
    
    def is_available(self) -> bool:
        if not self._available:
            return False
        
        try:
            # Quick health check
            self.client.text_generation(prompt="test", model=self.model, max_new_tokens=5)
            return True
        except:
            return False
        
class GroqProvider(LLMProvider):
    """Groq Inference API provider."""

    def __init__(self, api_key: str, model: str = "groq/groq-llm"):
        self.api_key = api_key
        self.model = model
        self.client = Groq(api_key=self.api_key)
        self._available = bool(api_key)

    def generate_response(self, prompt: str, **kwargs) -> str:
        try:
            response = self.client.chat.completions.create(
                messages=[
                    {"role": "user", 
                     "content": prompt}
                ],
                model=self.model,
                max_new_tokens=kwargs.get("max_new_tokens", 256),
                temperature=kwargs.get("temperature", 0.3),
                top_p=kwargs.get("top_p", 0.9)
            )
            return response.choices[0].message.content
        
        except Exception as e:
            print(f"Error generating response from Groq: {e}")
            raise
    
    def is_available(self) -> bool:
        if not self._available:
            return False
        
        try:
            # Quick health check
            self.client.chat.completions.create(
                messages=[{"role": "user", "content": "test"}],
                model=self.model,
                max_new_tokens=5
            )
            return True
        except:
            return False
        
class MultiLLMManager:
    """Manage multiple LLM providers with automatic failover."""

    def __init__(self, providers: List[LLMProvider]):
        self.providers = providers
        self.current_provider_index = 0
        self.failover_history = []
        self.usage_stats = {provider.__class__.__name__: 0 for provider in providers}

    def generate_with_failover(self, prompt: str, **kwargs) -> Dict[str, Any]:
        """Generate response with automatic failover across providers."""
        start_time = time.time()
        errors = []

        # Try provider in order
        for i in range(len(self.providers)):
            idx = (self.current_provider_index + i) % len(self.providers)
            provider = self.providers[idx]

            try:
                print(f"🔍 Trying {provider.__class__.__name__}...")

                if not provider.is_available():
                    print(f"❌ {provider.__class__.__name__} unavailable")
                    continue

                response = provider.generate_response(prompt, **kwargs)

                # Success
                self.current_provider_index = idx
                self.usage_stats[provider.__class__.__name__] += 1

                return {
                    "response": response,
                    "provider": provider.__class__.__name__,
                    "latency": time.time() - start_time,
                    "success": True
                }

            except Exception as e:
                error_msg = f"Error with {provider.__class__.__name__} failed: {str(e)}"
                print(f"❌ {error_msg}")
                errors.append(error_msg)
                continue

        # All providers failed
        return {
            "response": None,
            "provider": None,
            "latency": time.time() - start_time,
            "success": False,
            "errors": errors
        }
    
    def get_stats(self) -> Dict:
        """Get usage statistics"""
        return {
            'usage_stats': self.usage_stats,
            'total_requests': sum(self.usage_stats.values()),
            'current_provider': self.providers[self.current_provider_index].__class__.__name__
        }
    
# Initialize providers with API keys
providers = []

# Huggingface inference API
if HUGGINGFACE_API_KEY:
    providers.append(HuggingFaceProvider(api_key=HUGGINGFACE_API_KEY,
                                         model="mistralai/Mistral-7B-Instruct-v0.1"))

# Groq inference API    
if GROQ_API_KEY:
    providers.append(GroqProvider(api_key=GROQ_API_KEY, 
                                  model="llama-3.3-70b-versatile"))
    
# Create multi LLM manager
llm_manager = MultiLLMManager(providers=providers)

print("\n🚀 Multi-LLM Manager Initialized:")
print(f"  Active Providers: {len(providers)}")
for provider in providers:
    print(f"  - {provider.__class__.__name__}: {'✅' if provider.is_available() else '❌'}")

## LangChain Integration for Structured Analysis

In [ ]:
# Legacy chains AND structured parsers both belong to langchain_classic now
from langchain_classic.chains import LLMChain, SequentialChain
from langchain_classic.output_parsers.structured import (
    ResponseSchema,
    StructuredOutputParser,
)

# Core templates and basic callback handlers stay in langchain_core
from langchain_core.callbacks import StreamingStdOutCallbackHandler
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

In [ ]:
class LangChainAbuseAnalyzer:
    """LangChain-based abuse analysis with structured output"""

    def __init__(self, llm_manager: MultiLLMManager):
        self.llm_manager = llm_manager
        self.chains = self._create_chains()

    def _create_chains(self):
        """Create LangChain chains for analysis"""

        # 1. Abuse Pattern Detection Chain
        pattern_prompt = PromptTemplate(
            input_variables=["transaction_data", "abuse_core", "patterns"],
            template = """
            You are a fruad detection expert. Analyze this transaction for abuse patterns.

            Transaction Data: {transaction_data}
            Abuse Core: {abuse_core}
            Detected Patterns: {patterns}

            Provide a structured analysis with:
            1. Primary abuse pattern identified
            2. Confidence level (0-100)
            3. Severity assessment (LOW/MEDIUM/HIGH/CRITICAL)
            4. Specific indicators found

            Format response as JSON with keys: pattern, confidence, severity, indicators
            """
        )

        # 2. Investigation Chain
        investigation_prompt = PromptTemplate(
            input_variables=["analysis", "historical_context"],
            template = """
            You are a security investigator. Based on the abuse analysis, conduct a thorough investigation.

            Analysis Results: {analysis}
            Historical Context: {historical_context}

            Provide:
            1. Root cause analysis
            2. Potential impact assessment
            3. Immediate actions needed
            4. Long-term recommendations

            Respond in structured JSON format.
            """
        )

        # 3. Prevention Chain
        prevention_prompt = PromptTemplate(
            input_variables=["investigation", "risk_level"],
            template="""
            Based on the investigation findings, recommend prevention strategies.
            
            Investigation: {investigation}
            Risk Level: {risk_level}
            
            Provide:
            1. Short-term prevention measures (next 24 hours)
            2. Medium-term improvements (next 30 days)
            3. Long-term strategic changes
            4. ML model improvement suggestions
            
            Format as JSON.
            """
        )

        # Create LLM Chains
        pattern_chain = LLMChain(
            llm=self._get_langchain_llm(),
            prompt=pattern_prompt,
            output_key="pattern_analysis",
            verbose=False
        )

        investigation_chain = LLMChain(
            llm=self._get_langchain_llm(),
            prompt=investigation_prompt,
            output_key="investigation",
            verbose=False
        )

        prevention_chain = LLMChain(
            llm=self._get_langchain_llm(),
            prompt=prevention_prompt,
            output_key="prevention",
            verbose=False
        )

        # Create sequential chain
        overall_chain = SequentialChain(
            chains=[pattern_chain, investigation_chain, prevention_chain],
            input_variables=["transaction_data", "abuse_core", "patterns", "historical_context", "risk_level"],
            output_variables=["pattern_analysis", "investigation", "prevention"],
            verbose=False
        )

        return overall_chain
    
    def _get_langchain_llm(self):
        """Get LangChain-compatible LLM wrapper using the available provider"""

        class LangChainLLMWrapper:
            def __init__(self, manager):
                self.manager = manager

            def __call__(self, prompt, **kwargs):
                result = self.manager.generate_with_failover(prompt, **kwargs)

                if result["success"]:
                    return result["response"]
                raise Exception(f"All LLM providers failed: {result.get('errors', [])}")
            
        return LangChainLLMWrapper(self.llm_manager)
    
    def analyze_abuse(self, transaction_data: Dict, abuse_score: float,
                      patterns: List[str], historical_context: str = "") -> Dict:
        """Run complete Langchain analysis pipeline"""

        try:
            # Run the sequential chain
            result = self.chains({
                "transaction_data": json.dumps(transaction_data, indent=2),
                "abuse_core": abuse_score,
                "patterns": ", ".join(patterns),
                "historical_context": historical_context or "No historical context available.",
                "risk_level": self._get_risk_level(abuse_score)
            })

            # Parse JSON responses
            result['pattern_analysis'] = json.loads(result['pattern_analysis'])
            result['investigation'] = json.loads(result['investigation'])
            result['prevention'] = json.loads(result['prevention'])

            return result
        
        except Exception as e:
            print(f"⚠️ Langchain analysis error: {e}")
            return self._fallback_analysis(transaction_data, abuse_score, patterns)
        
    def _get_risk_level(self, abuse_score: float) -> str:
        """Categorize risk level"""
        if abuse_score >= 0.8:
            return "CRITICAL"
        elif abuse_score >= 0.6:
            return "HIGH"
        elif abuse_score >= 0.4:
            return "MEDIUM"
        else:
            return "LOW"
        
    def _fallback_analysis(self, transaction_data: Dict, abuse_score: float, patterns: List[str]):
        """Fallback if LangChain fails"""
        return {
            "pattern_analysis": {
                "pattern": "Unknown - Fallback mode",
                "confidence": 50,
                "severity": self._get_risk_level(abuse_score),
                "indicators": patterns
            },
            "investigation": {
                "root_cause": "Unable to determine - manual review recommended",
                "impact": "Unknown",
                "immediate_actions": ["Manual investigation required"],
                "long_term": "Review model thresholds"
            },
            "prevention": {
                "short_term": ["Monitor account activity"],
                "medium_term": ["Update fraud rules"],
                "long_term": ["Improve model with new data"],
                "ml_improvements": ["Add more features for detection"]
            }
        }
    
# Initialize LangChain Abuse Analyzer
print("🔗 Initializing LangChain Abuse Analyzer...")
langchain_analyzer = LangChainAbuseAnalyzer(llm_manager=llm_manager)
print("✅ LangChain analyzer ready")

## Enhanced Streamlit Integration with Multi-LLM

In [ ]:
import streamlit as st
import pandas as pd
import json
from typing import Dict, Any, Optional

class EnchancedStreamlitDetector:
    """Enchanced detector with multi-LLM support for streamlit applications"""

    def __init__(self, model, scaler, feature_importance, llm_manager, langchain_analyzer):
        self.model = model
        self.scaler = scaler
        self.feature_importance = feature_importance
        self.llm_manager = llm_manager
        self.langchain_analyzer = langchain_analyzer

        # Threat patterns
        self.threat_patterns = [
            {'name': 'Bulk Discount Abuse', 'threshold': 0.7},
            {'name': 'High Velocity Orders', 'threshold': 0.6},
            {'name': 'New Customer High Spender', 'threshold': 0.8},
            {'name': 'Negative Profit Margin', 'threshold': 0.5},
            {'name': 'Location Inconsistency', 'threshold': 0.65},
        ]

    def predict_with_analysis(self, input_data: Dict) -> Dict[str, Any]:
        """Predict abuse with LLM-powered analysis"""

        # 1. Get base prediction
        input_df = pd.DataFrame([input_data])
        input_features = self._preprocess_features(input_df)
        input_scaled = self.scaler.transform(input_features)

        # 2. Prediction
        prediction = self.model.predict(input_scaled)
        probabilities = self.model.predict_proba(input_scaled)
        abuse_score = float(probabilities[0][1]) if len(probabilities[0]) > 1 else 0.5

        # 3. Pattern analysis
        pattern_analysis = self._analyze_pattern(input_data, abuse_score)
        risk_level = self._categorize_risk(abuse_score)

        # 4. LLM investigation (if risk HIGH or CRITICAL)
        llm_investigation = None
        if risk_level in ['HIGH', 'CRITICAL']:
            try:
                # Use Langchain for structured analysis
                llm_result = self.langchain_analyzer.analyze_abuse(
                    transaction_data=input_data,
                    abuse_score=abuse_score,
                    patterns=pattern_analysis,
                    historical_context=self._get_historical_context(input_data)
                )
                llm_investigation = llm_result

            except Exception as e:
                print(f"⚠️ LLM investigation failed: {e}")
                # Fallback to simple LLM investigation
                llm_investigation = self._simple_llm_investigation(input_data, abuse_score)

        return {
            'abuse_prediction': int(prediction[0]),
            'abuse_score': abuse_score,
            'confidence': float(np.max(probabilities)),
            'risk_level': risk_level,
            'pattern_analysis': pattern_analysis,
            'llm_investigation': llm_investigation,
            'recommendation': self._generate_recommendation(risk_level, pattern_analysis),
            'llm_stats': self.llm_manager.get_stats()
        }
    
    def _prepare_input(self, df: pd.DataFrame):
        """Prepare input features"""
        numeric_cols = self.feature_importance['feature'].values
        df_numeric = df[numeric_cols] if all(col in df.columns for col in numeric_cols) else df.select_dtypes(include=[np.number])
        
        return df_numeric
    
    def _analyze_pattern(self, input_data: Dict, abuse_score: float) -> List[str]:
        """Identify abuse patterns"""
        patterns_detected = []
        
        if input_data.get('discount_abuse_score', 0) >= 0.4:
            patterns_detected.append('Discount Abuse')
        if input_data.get('monthly_orders', 0) > 5:
            patterns_detected.append('High Transaction Velocity')
        if input_data.get('profit_margin_anomaly', 0) == 1:
            patterns_detected.append('Negative Profit Margin')
        if input_data.get('state_abuse', 0) == 1:
            patterns_detected.append('Suspicious Location Pattern')
        
        for pattern in self.threat_patterns:
            if abuse_score > pattern['threshold']:
                patterns_detected.append(pattern['name'])
        
        return list(set(patterns_detected))
    
    def _categorize_risk(self, abuse_score: float) -> str:
        if abuse_score >= 0.8:
            return 'CRITICAL'
        elif abuse_score >= 0.6:
            return 'HIGH'
        elif abuse_score >= 0.4:
            return 'MEDIUM'
        else:
            return 'LOW'
    
    def _get_historical_context(self, input_data: Dict) -> str:
        """Get historical context for LLM analysis"""
        return f"Customer {input_data.get('customer_name', 'Unknown')} has {input_data.get('monthly_orders', 0)} orders this month"